# CatPred: compare inference speed

The outputs below are saved from a G4 run on 9 October 2026. You can read them without running any cells. [Download the saved results](https://github.com/Raalmonk/CatPred/raw/refs/heads/g4-inference/acceleration/colab/example_results/results.zip).

Choose **Runtime > Change runtime type > G4 GPU**, then run the cells in order. Setup checks the G4 Python 3.13 and PyTorch 2.11.0+cu130 environment.

Run the included public example, enter an enzyme sequence and reactant SMILES as in the [original demo](https://github.com/maranasgroup/CatPred/blob/main/colab_demo.ipynb), or upload a CSV. This notebook compares the original Python path with S_STREAM and adds K1 when the runtime supports it. All paths use the same ten kcat models, FP32, ESM features, batch size 50 and uncertainty calculations.

Raw member outputs and predictions must match exactly before timing starts. The table shows warm requests; ESM feature generation and model loading are measured separately.


In [1]:
#@title 1. Set up CatPred and cache the weights
use_drive_cache = True #@param {type:"boolean"}
pinned_commit = "" #@param {type:"string"}
#@markdown Leave the commit blank to use the fork's `g4-inference` branch.

from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import os
import re
import subprocess
import sys
import uuid

from IPython.display import display

HEADLESS = os.environ.get("CATPRED_NOTEBOOK_HEADLESS") == "1"
checkout_override = os.environ.get("CATPRED_NOTEBOOK_CHECKOUT")
CHECKOUT = Path(checkout_override or "/content/CatPred-compare")
WORK_ROOT = Path(os.environ.get("CATPRED_NOTEBOOK_WORK_ROOT", "/content/catpred-compare-runtime"))
REPOSITORY = "https://github.com/Raalmonk/CatPred.git"
SETUP_LOG = Path("/content") / ("catpred-setup-" + uuid.uuid4().hex[:8] + ".log")

def run_logged(arguments, log_path, *, cwd=None):
    with Path(log_path).open("w") as log:
        completed = subprocess.run(
            [str(value) for value in arguments], cwd=cwd,
            stdout=log, stderr=subprocess.STDOUT, text=True,
        )
    if completed.returncode:
        tail = Path(log_path).read_text(errors="replace")[-6000:]
        print(tail)
        raise RuntimeError(f"Stopped. Full log: {log_path}")

commit = pinned_commit.strip()
if commit and not re.fullmatch(r"[0-9a-fA-F]{40}", commit):
    raise ValueError("Use a full 40-character commit, or leave it blank.")
SOURCE_MANIFEST_SHA = None
SOURCE_BUNDLE_SHA = None
if CHECKOUT.exists() and not (CHECKOUT / ".git").exists():
    if not checkout_override:
        raise RuntimeError(f"{CHECKOUT} already exists and is not a Git checkout.")
    if commit:
        raise RuntimeError("A pinned commit needs a Git checkout.")
    bundle_path = CHECKOUT / "source_bundle_manifest.json"
    bundle_manifest = json.loads(bundle_path.read_text())
    bundle_records = bundle_manifest.get("files", [])
    base_commit = bundle_manifest.get("checkout_commit", "")
    if (bundle_manifest.get("schema") != 1 or not bundle_records
            or not re.fullmatch(r"[0-9a-fA-F]{40}", base_commit)):
        raise RuntimeError("Invalid uploaded source bundle manifest.")
    seen_bundle_paths = set()
    for record in bundle_records:
        relative = Path(record["path"])
        source_path = CHECKOUT / relative
        if (relative.is_absolute() or ".." in relative.parts or source_path.is_symlink()
                or not source_path.resolve().is_relative_to(CHECKOUT.resolve())
                or str(relative) in seen_bundle_paths):
            raise RuntimeError("Invalid source bundle path.")
        seen_bundle_paths.add(str(relative))
        if hashlib.sha256(source_path.read_bytes()).hexdigest() != record["sha256"]:
            raise RuntimeError("Uploaded source differs from bundle: " + str(relative))
    required_bundle_paths = {"colab_compare.ipynb", "demo/batch_kcat.csv",
        "acceleration/SOURCE_MANIFEST.json", "acceleration/colab/setup_runtime.py",
        "acceleration/colab/compare.py"}
    if not required_bundle_paths.issubset(seen_bundle_paths):
        raise RuntimeError("The source bundle is missing comparison files.")
    SOURCE_BUNDLE_SHA = hashlib.sha256(bundle_path.read_bytes()).hexdigest()
    acceleration = (CHECKOUT / "acceleration").resolve()
    manifest_path = acceleration / "SOURCE_MANIFEST.json"
    source_manifest = json.loads(manifest_path.read_text())
    source_records = source_manifest.get("copied_files", [])
    if not source_records:
        raise RuntimeError("The uploaded checkout has no source manifest entries.")
    seen_source_paths = set()
    for record in source_records:
        relative = Path(record["path"])
        source_path = acceleration / relative
        if (relative.is_absolute() or ".." in relative.parts or source_path.is_symlink()
                or not source_path.resolve().is_relative_to(acceleration)
                or str(relative) in seen_source_paths):
            raise RuntimeError("Invalid source manifest path.")
        seen_source_paths.add(str(relative))
        if hashlib.sha256(source_path.read_bytes()).hexdigest() != record["sha256"]:
            raise RuntimeError("Uploaded source differs from manifest: " + str(relative))
    SOURCE_MANIFEST_SHA = hashlib.sha256(manifest_path.read_bytes()).hexdigest()
    ACTUAL_COMMIT = base_commit
    CHECKOUT_STATUS = "Uploaded source snapshot; SOURCE_MANIFEST verified"
else:
    if CHECKOUT.exists():
        origin = subprocess.check_output(
            ["git", "-C", str(CHECKOUT), "remote", "get-url", "origin"], text=True
        ).strip().rstrip("/")
        if origin not in {REPOSITORY, REPOSITORY.removesuffix(".git"), "git@github.com:Raalmonk/CatPred.git"}:
            raise RuntimeError("The existing checkout has a different origin. Use a fresh Colab session.")
        actual_commit = subprocess.check_output(
            ["git", "-C", str(CHECKOUT), "rev-parse", "HEAD"], text=True
        ).strip()
        if commit and actual_commit.lower() != commit.lower():
            raise RuntimeError("The existing checkout is at a different commit. Its files were kept.")
    else:
        run_logged(["git", "clone", "--branch", "g4-inference", REPOSITORY, CHECKOUT], SETUP_LOG)
        if commit:
            run_logged(["git", "-C", CHECKOUT, "checkout", "--detach", commit], SETUP_LOG)
    ACTUAL_COMMIT = subprocess.check_output(
        ["git", "-C", str(CHECKOUT), "rev-parse", "HEAD"], text=True
    ).strip()
    CHECKOUT_STATUS = subprocess.check_output(
        ["git", "-C", str(CHECKOUT), "status", "--porcelain"], text=True
    ).strip()
SETUP_SCRIPT = CHECKOUT / "acceleration/colab/setup_runtime.py"
COMPARE_SCRIPT = CHECKOUT / "acceleration/colab/compare.py"
if not SETUP_SCRIPT.is_file() or not COMPARE_SCRIPT.is_file():
    raise RuntimeError("This checkout does not contain the comparison scripts. Use a fresh Colab session.")

asset_cache_override = os.environ.get("CATPRED_NOTEBOOK_ASSET_CACHE")
if asset_cache_override:
    ASSET_CACHE = Path(asset_cache_override)
elif use_drive_cache and not HEADLESS:
    from google.colab import drive
    drive.mount("/content/drive")
    ASSET_CACHE = Path("/content/drive/MyDrive/CatPred/weights-cache")
else:
    ASSET_CACHE = Path("/content/catpred-weights-cache")

print("Installing dependencies and fetching any missing weights...")
run_logged(
    [sys.executable, SETUP_SCRIPT, "--checkout", CHECKOUT,
     "--work-root", WORK_ROOT, "--asset-cache", ASSET_CACHE, "--run"],
    SETUP_LOG,
)
SETUP_PATH = WORK_ROOT / "setup.json"
SETUP = json.loads(SETUP_PATH.read_text())
if SETUP.get("status") != "ready":
    raise RuntimeError("Setup did not finish. See " + str(SETUP_LOG))
PYTHON = SETUP["python"]
print("Ready. GPU:", SETUP["hardware"]["gpu"])
if ACTUAL_COMMIT:
    print("Checkout:", ACTUAL_COMMIT)
else:
    print("Source manifest:", SOURCE_MANIFEST_SHA)
if SOURCE_BUNDLE_SHA:
    print("Uploaded source bundle:", SOURCE_BUNDLE_SHA)
print("Weights cache:", ASSET_CACHE)


Installing dependencies and fetching any missing weights...


Ready. GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
Checkout: b63cf6522c8b49555e60f42f843de7659e0a4cd5
Uploaded source bundle: 2c20f9f951912404b10d40688c42c76df319f42f785c09cd20cecdd7985fd9b7
Weights cache: /content/catpred_colab_validation/asset-cache


## Input

The default uses the 14 public rows in `demo/batch_kcat.csv`, repeated 8 times for 112 rows. Change `repeat_count` to adjust the example size. All paths evaluate every row; prediction caching and row deduplication are disabled.

The form uses the original notebook's example. For kcat, join all reactants with `.` in the SMILES field. An uploaded CSV needs `sequence` and `SMILES` columns; its row order is preserved and the repeat setting does not apply. Maximum: 16,384 rows.


In [2]:
#@title 2. Enter an example or upload a CSV
input_mode = "Bundled example" #@param ["Bundled example", "Form example", "Upload CSV"]
uniprot_id = 'P35557' #@param {type:"string"}
sequence = 'MLDDRARMEAAKKEKVEQILAEFQLQEEDLKKVMRRMQKEMDRGLRLETHEEASVKMLPTYVRSTPEGSEVGDFLSLDLGGTNFRVMLVKVGEGEEGQWSVKTKHQMYSIPEDAMTGTAEMLFDYISECISDFLDKHQMKHKKLPLGFTFSFPVRHEDIDKGILLNWTKGFKASGAEGNNVVGLLRDAIKRRGDFEMDVVAMVNDTVATMISCYYEDHQCEVGMIVGTGCNACYMEEMQNVELVEGDEGRMCVNTEWGAFGDSGELDEFLLEYDRLVDESSANPGQQLYEKLIGGKYMGELVRLVLLRLVDENLLFHGEASEQLRTRGAFETRFVSQVESDTGDRKQIYNILSTLGLRPSTTDCDIVRRACESVSTRAAHMCSAGLAGVINRMRESRSEDVMRITVGVDGSVYKLHPSFKERFHASVRRLTPSCEITFIESEEGSGRGAALVSAVACKKACMLGQ' #@param {type:"string"}
SMILES = 'C([C@@H]1[C@H]([C@@H]([C@H](C(O1)O)O)O)O)O.C1=NC(=C2C(=N1)N(C=N2)[C@H]3[C@@H]([C@@H]([C@H](O3)COP(=O)(O)OP(=O)(O)OP(=O)(O)O)O)O)N ' #@param {type:"string"}
repeat_count = 8 #@param {type:"integer"}

import csv
import io

if "SETUP" not in globals() or SETUP.get("status") != "ready":
    raise RuntimeError("Run setup first.")
if input_mode in {"Bundled example", "Form example"}:
    if type(repeat_count) is not int or not 1 <= repeat_count <= 16384:
        raise ValueError("Choose a positive repeat count, up to 16,384 total rows.")
    if input_mode == "Bundled example":
        with (CHECKOUT / "demo/batch_kcat.csv").open(newline="") as source:
            source_reader = csv.DictReader(source)
            fieldnames = source_reader.fieldnames
            example_rows = list(source_reader)
        input_description = f"{len(example_rows)} public example rows repeated {repeat_count} times ({len(example_rows) * repeat_count} rows)"
    else:
        fieldnames = ["name", "sequence", "SMILES", "pdbpath"]
        example_rows = [{"name": uniprot_id, "sequence": sequence, "SMILES": SMILES, "pdbpath": "example.pdb"}]
        input_description = f"The same form example repeated {repeat_count} times ({repeat_count} rows)"
    if not example_rows or len(example_rows) * repeat_count > 16384:
        raise ValueError("Use 1 to 16,384 total example rows.")
    buffer = io.StringIO(newline="")
    writer = csv.DictWriter(buffer, fieldnames=fieldnames)
    writer.writeheader()
    for _ in range(repeat_count):
        writer.writerows(example_rows)
    input_bytes = buffer.getvalue().encode("utf-8")
elif input_mode == "Upload CSV":
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload one CSV file.")
    filename, input_bytes = next(iter(uploaded.items()))
    if not filename.lower().endswith(".csv"):
        raise ValueError("Upload a .csv file.")
    input_description = "Uploaded CSV: " + Path(filename).name
else:
    raise ValueError("Choose Bundled example, Form example or Upload CSV.")

reader = csv.DictReader(io.StringIO(input_bytes.decode("utf-8-sig"), newline=""))
if not {"sequence", "SMILES"}.issubset(reader.fieldnames or []):
    raise ValueError("The CSV needs sequence and SMILES columns.")
row_count = sum(1 for _ in reader)
if not 1 <= row_count <= 16384:
    raise ValueError("The CSV must contain 1 to 16,384 rows.")

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
RUN_DIR = Path("/content/catpred-comparisons") / run_id
RUN_DIR.mkdir(parents=True, exist_ok=False)
for stale_name in ("summary", "OUTPUT_DIR", "SUMMARY_PATH", "COMPARE_LOG"):
    globals().pop(stale_name, None)
INPUT_CSV = RUN_DIR / "input.csv"
INPUT_CSV.write_bytes(input_bytes)
NOTEBOOK_CONFIG = {
    "checkout_commit": ACTUAL_COMMIT, "checkout_status": CHECKOUT_STATUS,
    "source_manifest_sha256": SOURCE_MANIFEST_SHA,
    "source_bundle_manifest_sha256": SOURCE_BUNDLE_SHA,
    "input_description": input_description, "input_rows": row_count,
    "input_sha256": hashlib.sha256(input_bytes).hexdigest(),
    "setup": str(SETUP_PATH), "parameter": "kcat",
}
(RUN_DIR / "notebook_config.json").write_text(json.dumps(NOTEBOOK_CONFIG, indent=2) + "\n")
print(input_description)
print(f"{row_count:,} rows saved to {INPUT_CSV}")


14 public example rows repeated 8 times (112 rows)
112 rows saved to /content/catpred-comparisons/20261009T142733Z-bbef9c68/input.csv


In [3]:
#@title 3. Check outputs and compare warm requests
repeats = 3 #@param {type:"integer"}

if type(repeats) is not int or not 1 <= repeats <= 5:
    raise ValueError("Choose 1 to 5 timing repetitions.")
if "INPUT_CSV" not in globals() or not INPUT_CSV.is_file():
    raise RuntimeError("Create or upload the input first.")
summary = {}
OUTPUT_DIR = RUN_DIR / ("comparison-" + uuid.uuid4().hex[:8])
COMPARE_LOG = RUN_DIR / (OUTPUT_DIR.name + ".log")
print("Generating ESM features, checking exact outputs, then timing each path...")
command = [
    PYTHON, COMPARE_SCRIPT, "--setup", SETUP_PATH, "--input", INPUT_CSV,
    "--output", OUTPUT_DIR, "--repeats", str(repeats), "--k1", "auto",
]
try:
    run_logged(command, COMPARE_LOG)
except RuntimeError:
    summary_path = OUTPUT_DIR / "summary.json"
    if summary_path.is_file():
        failed = json.loads(summary_path.read_text())
        print("Comparison:", failed.get("status", "FAILED"))
        if failed.get("reason") or failed.get("error"):
            print(failed.get("reason") or failed.get("error"))
    raise

SUMMARY_PATH = OUTPUT_DIR / "summary.json"
summary = json.loads(SUMMARY_PATH.read_text())
if summary.get("status") != "COMPLETE":
    print("Comparison:", summary.get("status", "INCOMPLETE"))
    if summary.get("reason") or summary.get("error"):
        print(summary.get("reason") or summary.get("error"))
    raise RuntimeError("No timing table was published. See " + str(SUMMARY_PATH))

import pandas as pd

print(f"Exact output checks passed for the timed paths ({summary['rows']:,} rows).")
hardware = summary.get("hardware", {})
print("GPU:", hardware.get("gpu", hardware.get("name", "See summary.json")))
preparation = summary["preparation"]
print(f"ESM features: {preparation['esm_seconds']:.3f} s for {preparation['unique_sequences']:,} unique sequences")
print(f"Model loading: {preparation['model_load_seconds']:.3f} s")

timing_rows = []
for arm, result in summary["arms"].items():
    if result.get("median_seconds") is not None:
        timing_rows.append({
            "Path": arm, "Median seconds": result["median_seconds"],
            "Speedup vs original": result.get("speedup_vs_original"),
        })
    else:
        print(f"{arm}: {result.get('status', 'not timed')}. {result.get('reason', '')}".strip())
if timing_rows:
    display(pd.DataFrame(timing_rows).style.format({"Median seconds": "{:.3f}", "Speedup vs original": "{:.2f}×"}, na_rep=""))
print(summary["timing_boundary"])
print("Saved results:", OUTPUT_DIR)


Generating ESM features, checking exact outputs, then timing each path...


Exact output checks passed for the timed paths (112 rows).
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
ESM features: 3.849 s for 9 unique sequences
Model loading: 0.336 s


,Path,Median seconds,Speedup vs original
0,Original,1.202,1.00×
1,S_STREAM,0.428,2.81×
2,S_STREAM_K1,0.393,3.06×


Warm complete request, synchronized, including preprocessing, all ten models, uncertainty, CSV and request-cache release. Model loading, ESM generation, activation checks and evidence serialization are outside this timer.
Saved results: /content/catpred-comparisons/20261009T142733Z-bbef9c68/comparison-eb2fcc42


In [4]:
#@title 4. View predictions
if "summary" not in globals() or summary.get("status") != "COMPLETE":
    raise RuntimeError("Finish the comparison first.")
for arm, result in summary["arms"].items():
    if result.get("prediction_path"):
        prediction_path = Path(result["prediction_path"])
        if not prediction_path.is_absolute():
            prediction_path = OUTPUT_DIR / prediction_path
        print(arm)
        frame = pd.read_csv(prediction_path)
        visible_columns = [name for name in ("row_id", "Substrate", "name", "Prediction_(s^(-1))", "Prediction_log10", "SD_total", "SD_aleatoric", "SD_epistemic") if name in frame.columns]
        display(frame.loc[:, visible_columns].head(10))


Original


,row_id,Substrate,Prediction_(s^(-1)),Prediction_log10,SD_total,SD_aleatoric,SD_epistemic
0,row_000000,Tyrosine,1.540091,0.187546,0.886527,0.633247,0.620425
1,row_000001,Tyrosine,2.196005,0.341633,0.857462,0.496443,0.699132
2,row_000002,Tyrosine,1.079495,0.033220,0.755585,0.492027,0.573426
3,row_000003,Tyrosine,5.823674,0.765197,0.531741,0.416962,0.329986
4,row_000004,Tyrosine,3.291237,0.517359,0.562111,0.474382,0.301547
5,row_000005,Tyrosine,5.137666,0.710766,0.522594,0.466109,0.236320
6,row_000006,Tyrosine,3.847815,0.585214,0.679737,0.383326,0.561341
7,row_000007,Tyrosine,3.607887,0.557253,0.745130,0.484061,0.566483
8,row_000008,pNP-β-GlcA,24.274714,1.385154,0.853352,0.533479,0.666041
9,row_000009,pNP-β-Glc,30.661644,1.486595,0.710590,0.383808,0.598022


S_STREAM


,row_id,Substrate,Prediction_(s^(-1)),Prediction_log10,SD_total,SD_aleatoric,SD_epistemic
0,row_000000,Tyrosine,1.540091,0.187546,0.886527,0.633247,0.620425
1,row_000001,Tyrosine,2.196005,0.341633,0.857462,0.496443,0.699132
2,row_000002,Tyrosine,1.079495,0.033220,0.755585,0.492027,0.573426
3,row_000003,Tyrosine,5.823674,0.765197,0.531741,0.416962,0.329986
4,row_000004,Tyrosine,3.291237,0.517359,0.562111,0.474382,0.301547
5,row_000005,Tyrosine,5.137666,0.710766,0.522594,0.466109,0.236320
6,row_000006,Tyrosine,3.847815,0.585214,0.679737,0.383326,0.561341
7,row_000007,Tyrosine,3.607887,0.557253,0.745130,0.484061,0.566483
8,row_000008,pNP-β-GlcA,24.274714,1.385154,0.853352,0.533479,0.666041
9,row_000009,pNP-β-Glc,30.661644,1.486595,0.710590,0.383808,0.598022


S_STREAM_K1


,row_id,Substrate,Prediction_(s^(-1)),Prediction_log10,SD_total,SD_aleatoric,SD_epistemic
0,row_000000,Tyrosine,1.540091,0.187546,0.886527,0.633247,0.620425
1,row_000001,Tyrosine,2.196005,0.341633,0.857462,0.496443,0.699132
2,row_000002,Tyrosine,1.079495,0.033220,0.755585,0.492027,0.573426
3,row_000003,Tyrosine,5.823674,0.765197,0.531741,0.416962,0.329986
4,row_000004,Tyrosine,3.291237,0.517359,0.562111,0.474382,0.301547
5,row_000005,Tyrosine,5.137666,0.710766,0.522594,0.466109,0.236320
6,row_000006,Tyrosine,3.847815,0.585214,0.679737,0.383326,0.561341
7,row_000007,Tyrosine,3.607887,0.557253,0.745130,0.484061,0.566483
8,row_000008,pNP-β-GlcA,24.274714,1.385154,0.853352,0.533479,0.666041
9,row_000009,pNP-β-Glc,30.661644,1.486595,0.710590,0.383808,0.598022


In [5]:
#@title 5. Download results
save_zip_to_drive = False #@param {type:"boolean"}

import shutil
import zipfile
from IPython.display import FileLink

if "OUTPUT_DIR" not in globals() or not OUTPUT_DIR.exists():
    raise RuntimeError("Run the comparison first.")
archive_path = RUN_DIR / (OUTPUT_DIR.name + ".zip")
allowed_suffixes = {".csv", ".npz", ".json", ".log"}
excluded_directories = {"feature_cache", "shared_feature_cache", "torch", "weights", "weights-cache"}
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUTPUT_DIR.rglob("*")):
        relative = path.relative_to(OUTPUT_DIR)
        if (path.is_file() and path.suffix in allowed_suffixes
                and not excluded_directories.intersection(relative.parts)):
            archive.write(path, Path(OUTPUT_DIR.name) / relative)
    for path in (INPUT_CSV, RUN_DIR / "notebook_config.json", COMPARE_LOG):
        if path.is_file():
            archive.write(path, path.name)
    archive.write(SETUP_PATH, "setup.json")

if save_zip_to_drive:
    if not HEADLESS:
        from google.colab import drive
        drive.mount("/content/drive")
    if not Path("/content/drive/MyDrive").is_dir():
        raise RuntimeError("Mount Drive before saving the zip there.")
    destination = Path("/content/drive/MyDrive/CatPred/comparisons") / RUN_DIR.name
    destination.mkdir(parents=True, exist_ok=True)
    saved_archive = destination / archive_path.name
    shutil.copy2(archive_path, saved_archive)
    print("Saved to Drive:", saved_archive)
display(FileLink(str(archive_path), result_html_prefix="Results zip: "))
if not HEADLESS:
    import ipywidgets as widgets
    from google.colab import files
    download_button = widgets.Button(description="Download zip")
    def download_results(_button, download_path=str(archive_path)):
        files.download(download_path)
    download_button.on_click(download_results)
    display(download_button)


/content/catpred-comparisons/20261009T142733Z-bbef9c68/comparison-eb2fcc42.zip